# CF03 — strict numerical audit notebook

This notebook instantiates the **strict numerical gates template** for **CF03: A8 Scaling Theorem (Hierarchical Tachyonic Interfaces)**.

Contract:
- one real audit unit per **claim / theorem / proposition / falsifier gate / worked witness**
- every executable cell emits **one inline figure** and **one terminal pass/fail log**
- no filesystem output is required for correctness
- **A(-1)** is treated as **reference-only context for the generative engine**, not as a runtime dependency of the audit


## Required authoring model

Treat this notebook as an **auditor attacking the paper's burden**, not as an explainer.

That means:
1. every unit states **what would count as failure**
2. every unit includes a **numerical or structural attack**, not prose reassurance
3. worked examples are labeled as **witnesses / sanity attacks**, not promoted into proofs
4. empirical cells remain explicit about scope: they can support or weaken a claim, but they do not silently replace theorem-level rigor


## Gate T0 — Template substrate integrity

**Plain-language view**

Before touching CF03, the notebook must prove that its own substrate obeys the contract.

**What this cell attacks**
- missing imports
- no ledger
- helper stack missing
- setup cell that only dumps raw text/dicts
- hidden filesystem dependency

**Pass / fail criteria**
- imports available
- ledger initialized
- helper functions defined
- inline figure rendered
- terminal pass/fail log present
- filesystem I/O not required


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=10, suppress=True)

trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

@dataclass
class GateResult:
    gate_id: str
    gate_name: str
    passed: bool
    metrics: Dict[str, Any]
    pass_criteria: Dict[str, Any]
    notes: str

LEDGER: List[GateResult] = []

def terminal_log(gate_id: str, title: str, passed: bool, metrics: Dict[str, Any], criteria: Dict[str, Any], notes: str = "") -> None:
    status_symbol = "✅" if passed else "❌"
    print("=" * 100)
    print(f"[{gate_id}] {title}")
    print("- status:", f"{status_symbol} {'PASS' if passed else 'FAIL'}")
    print("- criteria:")
    for k, v in criteria.items():
        print(f"    * {k}: {v}")
    print("- metrics:")
    for k, v in metrics.items():
        print(f"    * {k}: {v}")
    if notes:
        print("- notes:", notes)
    print("=" * 100)

def W(phi: np.ndarray) -> np.ndarray:
    return 0.25 * (1.0 - phi**2) ** 2

C0_EXACT = 2.0 * np.sqrt(2.0) / 3.0

def heteroclinic_profile(x: np.ndarray, eps: float, center: float = 0.0) -> np.ndarray:
    return np.tanh((x - center) / (np.sqrt(2.0) * eps))

def multi_interface_profile(x: np.ndarray, centers: List[float], eps: float) -> np.ndarray:
    phi = np.ones_like(x)
    for c in centers:
        phi *= np.tanh((x - c) / (np.sqrt(2.0) * eps))
    return phi

def phase_energy_1d(x: np.ndarray, phi: np.ndarray, eps: float) -> Tuple[float, np.ndarray, np.ndarray]:
    dphi = np.gradient(phi, x)
    density = 0.5 * eps * dphi**2 + W(phi) / eps
    return float(trapz(density, x)), density, dphi

def perimeter_proxy_open(phi: np.ndarray) -> int:
    s = (phi >= 0.0).astype(int)
    return int(np.sum(np.abs(np.diff(s))))

def perimeter_proxy_periodic(phi: np.ndarray) -> int:
    s = (phi >= 0.0).astype(int)
    return int(np.sum(np.abs(np.diff(np.r_[s, s[0]]))))

def spectral_gaussian_smooth(phi: np.ndarray, dx: float, sigma: float) -> np.ndarray:
    if sigma <= 0.0:
        return phi.copy()
    k = 2.0 * np.pi * np.fft.fftfreq(len(phi), d=dx)
    filt = np.exp(-0.5 * (sigma * k) ** 2)
    return np.fft.ifft(np.fft.fft(phi) * filt).real

def dyadic_scales(L: float, ell0: float) -> np.ndarray:
    scales = []
    k = 0
    while L / (2**k) >= ell0 - 1e-12:
        scales.append(L / (2**k))
        k += 1
    return np.array(scales, dtype=float)

def active_boundary_census(phi: np.ndarray, L: float, ell0: float, dx: float, sigma_frac: float = 0.20) -> Tuple[np.ndarray, np.ndarray]:
    scales = dyadic_scales(L, ell0)
    counts = []
    for ell in scales:
        sm = spectral_gaussian_smooth(phi, dx, sigma=sigma_frac * ell)
        counts.append(perimeter_proxy_periodic(sm))
    return scales, np.array(counts, dtype=int)

def allen_cahn_step(phi: np.ndarray, dx: float, dt: float, eps: float) -> np.ndarray:
    lap = (np.roll(phi, -1) - 2.0 * phi + np.roll(phi, 1)) / dx**2
    phi_new = phi + dt * (eps**2 * lap - (phi**3 - phi))
    return np.clip(phi_new, -1.5, 1.5)

def linear_fit_metrics(x: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    coef = np.polyfit(x, y, 1)
    yhat = np.polyval(coef, x)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0
    return {"slope": float(coef[0]), "intercept": float(coef[1]), "r2": float(r2), "yhat": yhat}

def aic_bic(y: np.ndarray, yhat: np.ndarray, p: int) -> Tuple[float, float]:
    n = len(y)
    rss = float(np.sum((y - yhat) ** 2))
    rss = max(rss, 1e-15)
    aic = n * np.log(rss / n) + 2 * p
    bic = n * np.log(rss / n) + p * np.log(n)
    return float(aic), float(bic)

def fit_log_model(ratio: np.ndarray, y: np.ndarray) -> Dict[str, Any]:
    X = np.vstack([np.log2(ratio), np.ones_like(ratio)]).T
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    yhat = X @ coef
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0
    aic, bic = aic_bic(y, yhat, p=2)
    return {"coef": coef, "yhat": yhat, "r2": float(r2), "aic": aic, "bic": bic}

def fit_power_model(ratio: np.ndarray, y: np.ndarray) -> Dict[str, Any]:
    mask = (ratio > 0) & (y > 0)
    lx = np.log(ratio[mask])
    ly = np.log(y[mask])
    A = np.vstack([lx, np.ones_like(lx)]).T
    coef, *_ = np.linalg.lstsq(A, ly, rcond=None)
    alpha, beta = coef
    yhat = np.exp(beta) * ratio**alpha
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0
    aic, bic = aic_bic(y, yhat, p=2)
    return {"coef": np.array([alpha, beta]), "yhat": yhat, "r2": float(r2), "aic": aic, "bic": bic}

criteria = {
    "imports_available": True,
    "ledger_initialized": True,
    "helper_stack_defined": True,
    "inline_figure_rendered": True,
    "stdout_terminal_log_present": True,
    "filesystem_io_required": False,
}
metrics = {
    "imports_available": True,
    "ledger_length_after_init": len(LEDGER),
    "helper_stack_defined": all(name in globals() for name in [
        "terminal_log",
        "phase_energy_1d",
        "multi_interface_profile",
        "active_boundary_census",
        "fit_log_model",
        "fit_power_model",
    ]),
    "filesystem_io_required": False,
}

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.bar(
    ["imports", "ledger", "helpers", "inline fig", "stdout", "no I/O"],
    [1, 1 if metrics["ledger_length_after_init"] == 0 else 0, 1 if metrics["helper_stack_defined"] else 0, 1, 1, 1],
)
ax.set_ylim(0, 1.2)
ax.set_title("Template substrate contract check")
ax.set_ylabel("satisfied = 1")
ax.grid(alpha=0.25)
plt.show()

passed = (
    metrics["imports_available"]
    and metrics["ledger_length_after_init"] == 0
    and metrics["helper_stack_defined"]
    and not metrics["filesystem_io_required"]
)
terminal_log(
    "T0",
    "Template substrate integrity",
    passed,
    metrics,
    criteria,
    notes="Setup is audit-aware: helper definitions live here, and this setup cell still obeys the figure + gate-log contract.",
)
LEDGER.append(GateResult("T0", "Template substrate integrity", passed, metrics, criteria, "Strict template substrate verified."))


## Paper-spec contract

The next cell instantiates the paper manifest. It is intentionally concise: enough source text to anchor the audit, not a full paper dump.

The manifest must enumerate the real burden units:
- claim statements
- theorem / proposition statements
- falsifier gates
- worked witnesses treated as witnesses, not silently upgraded to proofs


## Gate T1 — Manifest completeness and burden coverage

**Plain-language view**

This attacks the most common failure mode in bad audit notebooks: they cover only the easy headline claims and ignore theorems, proposition statements, falsifier gates, or worked witnesses.

**Pass / fail criteria**
- manifest top-level fields present
- each burden unit has anchor, statement, pass criteria, fail signature, and output contract
- expected outputs include both a figure and a terminal log
- CF03 burden count is complete for this notebook's scope


In [ ]:
PAPER_SPEC = {
    "paper_id": "CF03",
    "paper_title": "A8 Scaling Theorem (Hierarchical Tachyonic Interfaces)",
    "external_reference_note": "A(-1) is reference-only and is not loaded by the notebook.",
    "paper_source_embedded": r'''CF03 title: A8 Scaling Theorem (Hierarchical Tachyonic Interfaces)

Embedded burden excerpt:
- C1: phase-field energy Gamma-converges to sharp-interface perimeter energy with c0 = ∫_{-1}^{1} sqrt(2 W(s)) ds.
- C2: excess energy concentrates in an O(eps) neighborhood of the interface.
- C3: active hierarchy depth obeys N(L) <= ceil(log2(L/ell0)) + 1, while hierarchical constructions can realize N(L) = Omega(log(L/ell0)).
- C4: under perimeter-reducing relaxation, same-scale interface multiplicities are suppressed rather than growing combinatorially.
- Theorem [Modica–Mortola]: E_eps Gamma-converges to E0 = c0 Per(A).
- Proposition [Logarithmic upper bound]: any dyadic-ladder activity definition gives N(L) <= K_max(L)+1.
- Gates G1-G4 are decisive falsifiers for perimeter scaling, multiplicity blow-up, depth model selection, and dimensionless collapse.
- Worked example: c0 = 2 sqrt(2)/3 for W=(1-s^2)^2/4, and a dyadic witness gives E0 ~ c0 N(L) = Theta(log(L/ell0)).''',
    "paper_validation_gates": [
        {"gate_id": "G1", "description": "Perimeter proxy scaling in the sharp-interface regime."},
        {"gate_id": "G2", "description": "No per-scale exponential growth of active multiplicity."},
        {"gate_id": "G3", "description": "Depth scales like log(L/ell0) and beats power-law alternatives."},
        {"gate_id": "G4", "description": "Curves collapse under the dimensionless ratio L/ell0."},
    ],
    "claims": [
        {
            "claim_id": "C1",
            "canon_anchor": "§1.1 / Claim C1",
            "plain_language_claim": "A resolved phase-field interface should cost the same limiting energy as one sharp interface, namely c0.",
            "claim_type": "scaling_bridge",
            "statement": "E_eps Gamma-converges to E0 = c0 Per(A) with c0 = ∫_-1^1 sqrt(2W(s)) ds.",
            "attack_mode": "convergence sweep + exact residual norm",
            "pass_criteria": {"max_relative_error": "<= 5e-5"},
            "fail_signature": "Resolved heteroclinic energies stay O(1) away from c0.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "THM_MM",
            "canon_anchor": "§2.2 / Theorem Modica–Mortola",
            "plain_language_claim": "If the theorem is the right effective bridge, total energy should scale linearly with boundary count for separated interfaces.",
            "claim_type": "theorem",
            "statement": "Phase-field energies Gamma-converge to c0 Per(A) for finite-perimeter sets.",
            "attack_mode": "parameter sweep + linearity residual + negative control",
            "pass_criteria": {"slope_relative_error": "<= 1e-3", "r2": ">= 0.999"},
            "fail_signature": "Energy ceases to be linear in boundary count.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "C2",
            "canon_anchor": "§3 / Claim C2",
            "plain_language_claim": "Almost all excess energy should live near the interface, not in the bulk.",
            "claim_type": "localization_claim",
            "statement": "Excess energy concentrates in a thin O(eps) layer around the phase boundary.",
            "attack_mode": "localization sweep + cumulative mass test",
            "pass_criteria": {"energy_fraction_inside_2eps": ">= 0.98"},
            "fail_signature": "Bulk regions retain a non-negligible fraction of excess energy.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "C3",
            "canon_anchor": "§4 / Claim C3",
            "plain_language_claim": "Depth cannot outrun the dyadic ladder, and a nested witness can still realize logarithmic growth.",
            "claim_type": "scaling_prediction",
            "statement": "N(L) <= ceil(log2(L/ell0))+1 and hierarchical constructions realize N(L)=Omega(log(L/ell0)).",
            "attack_mode": "model sweep + realizability witness + bound check",
            "pass_criteria": {"log_fit_r2": ">= 0.98", "bound_violations": 0},
            "fail_signature": "Depth grows materially faster than the ladder or the witness is not logarithmic.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "PROP_LOG",
            "canon_anchor": "§4.2 / Proposition Logarithmic upper bound",
            "plain_language_claim": "Any dyadic-ladder notion of active depth is capped by the number of available dyadic scales.",
            "claim_type": "proposition",
            "statement": "For any dyadic-ladder activity definition, N(L) <= K_max(L)+1.",
            "attack_mode": "exhaustive combinatorial enumeration + exact bound check",
            "pass_criteria": {"violations": 0},
            "fail_signature": "A dyadic activity mask yields depth larger than its ladder length.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "C4",
            "canon_anchor": "§5 / Claim C4",
            "plain_language_claim": "Perimeter-reducing relaxation should not spontaneously amplify same-scale boundary multiplicity on a hierarchical initial condition.",
            "claim_type": "empirical_claim",
            "statement": "Allen–Cahn / perimeter-reducing dynamics suppress combinatorial blow-up at fixed scale.",
            "attack_mode": "time evolution + per-scale census + monotonicity check",
            "pass_criteria": {"max_positive_count_increase": 0, "energy_nonincreasing": True},
            "fail_signature": "Relaxation creates new same-scale multiplicity or increases diffuse energy.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "G1",
            "canon_anchor": "§1.2 / Gate G1",
            "plain_language_claim": "A true perimeter proxy should line up with excess energy; a bad proxy should not.",
            "claim_type": "falsifier",
            "statement": "Fit E_exc ~ sigma * Per_hat with relative error <= 5 percent over a scale sweep.",
            "attack_mode": "proxy comparison + residual test + negative control",
            "pass_criteria": {"mean_relative_error": "<= 0.05", "negative_control_rejected": True},
            "fail_signature": "Perimeter proxy is not materially better than a non-perimeter proxy.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "G2",
            "canon_anchor": "§1.2 / Gate G2",
            "plain_language_claim": "The gate must pass a bounded hierarchical multiplicity sequence and fail an exponential per-scale sequence.",
            "claim_type": "falsifier",
            "statement": "For each scale k, active multiplicity m_k <= m_max independent of k.",
            "attack_mode": "gate separation test + adversarial exponential negative control",
            "pass_criteria": {"hierarchical_passes": True, "exponential_fails": True},
            "fail_signature": "The gate cannot distinguish bounded hierarchy from combinatorial blow-up.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "G3",
            "canon_anchor": "§1.2 / Gate G3",
            "plain_language_claim": "The gate must pick the logarithmic depth law on log-depth data and reject it on power-law data.",
            "claim_type": "falsifier",
            "statement": "N(L) fits a log model with R^2>=0.98 and beats power-law alternatives by AIC/BIC.",
            "attack_mode": "model selection + negative control",
            "pass_criteria": {"log_dataset_prefers_log": True, "power_dataset_rejects_log": True},
            "fail_signature": "Information criteria cannot separate logarithmic and power-law depth laws.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "G4",
            "canon_anchor": "§1.2 / Gate G4",
            "plain_language_claim": "If A8 is really a scaling law, curves should collapse by L/ell0 alone and fail to collapse when extra scale dependence is injected.",
            "claim_type": "falsifier",
            "statement": "Curves collapse when plotted against L/ell0 rather than separate dependence on L and ell0.",
            "attack_mode": "dimensionless collapse test + adversarial perturbation",
            "pass_criteria": {"true_collapse_error": "<= 0.02", "false_collapse_error": ">= 0.08"},
            "fail_signature": "Injected separate ell0 dependence does not measurably spoil collapse.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Direct",
        },
        {
            "claim_id": "W1",
            "canon_anchor": "§7.1 / Worked example single interface energy",
            "plain_language_claim": "For the specific quartic potential in CF03, c0 should evaluate exactly to 2*sqrt(2)/3.",
            "claim_type": "worked_witness",
            "statement": "c0 = ∫_-1^1 sqrt(1/2 * (1-s^2)^2) ds = 2*sqrt(2)/3.",
            "attack_mode": "exact quadrature residual",
            "pass_criteria": {"absolute_error": "<= 1e-10"},
            "fail_signature": "Numerical quadrature disagrees with the closed form.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Supporting witness",
        },
        {
            "claim_id": "W2",
            "canon_anchor": "§7.2 / Worked example nested construction",
            "plain_language_claim": "A one-interface-per-dyadic-scale witness should give log depth and energy proportional to that depth.",
            "claim_type": "worked_witness",
            "statement": "N(L) ~ floor(log2(L/ell0))+1 and E0 ~ c0 N(L) = Theta(log(L/ell0)).",
            "attack_mode": "constructive witness + fit residual",
            "pass_criteria": {"slope_relative_error": "<= 1e-12", "r2": ">= 0.999999"},
            "fail_signature": "Constructive witness fails to produce the advertised logarithmic law.",
            "expected_outputs": ["figure", "terminal_log", "result_metrics"],
            "paper_level_relevance": "Supporting witness",
        },
    ],
}

required_top = ["paper_id", "paper_title", "paper_source_embedded", "paper_validation_gates", "claims"]
required_claim = [
    "claim_id",
    "canon_anchor",
    "plain_language_claim",
    "claim_type",
    "statement",
    "attack_mode",
    "pass_criteria",
    "fail_signature",
    "expected_outputs",
    "paper_level_relevance",
]

top_missing = [k for k in required_top if k not in PAPER_SPEC]
claim_missing = {}
empty_statements = []
empty_plain_language = []
bad_outputs = []

for claim in PAPER_SPEC.get("claims", []):
    miss = [k for k in required_claim if k not in claim]
    if miss:
        claim_missing[claim.get("claim_id", "UNKNOWN")] = miss
    if not str(claim.get("statement", "")).strip():
        empty_statements.append(claim.get("claim_id", "UNKNOWN"))
    if not str(claim.get("plain_language_claim", "")).strip():
        empty_plain_language.append(claim.get("claim_id", "UNKNOWN"))
    expected = set(claim.get("expected_outputs", []))
    if not {"figure", "terminal_log"}.issubset(expected):
        bad_outputs.append(claim.get("claim_id", "UNKNOWN"))

metrics = {
    "top_missing_count": len(top_missing),
    "claim_missing_count": sum(len(v) for v in claim_missing.values()),
    "empty_statement_count": len(empty_statements),
    "empty_plain_language_count": len(empty_plain_language),
    "bad_output_contract_count": len(bad_outputs),
    "paper_validation_gate_count": len(PAPER_SPEC.get("paper_validation_gates", [])),
    "claim_count": len(PAPER_SPEC.get("claims", [])),
}
criteria = {
    "top_missing_count": 0,
    "claim_missing_count": 0,
    "empty_statement_count": 0,
    "empty_plain_language_count": 0,
    "bad_output_contract_count": 0,
    "paper_validation_gate_count": ">= 1",
    "claim_count": ">= 12",
}

fig, ax = plt.subplots(figsize=(8.2, 3.6))
bars = [
    metrics["claim_count"],
    metrics["paper_validation_gate_count"],
    metrics["top_missing_count"],
    metrics["claim_missing_count"],
    metrics["empty_plain_language_count"],
    metrics["bad_output_contract_count"],
]
ax.bar(
    ["claims", "paper gates", "top miss", "claim miss", "plain-lang miss", "bad outputs"],
    bars,
)
ax.set_title("Manifest completeness and CF03 burden coverage")
ax.grid(alpha=0.25)
plt.show()

passed = (
    metrics["top_missing_count"] == 0
    and metrics["claim_missing_count"] == 0
    and metrics["empty_statement_count"] == 0
    and metrics["empty_plain_language_count"] == 0
    and metrics["bad_output_contract_count"] == 0
    and metrics["paper_validation_gate_count"] >= 1
    and metrics["claim_count"] >= 12
)
terminal_log(
    "T1",
    "Manifest completeness and burden coverage",
    passed,
    metrics,
    criteria,
    notes="Coverage includes C1-C4, theorem, proposition, G1-G4, and the two worked witnesses W1-W2.",
)
LEDGER.append(GateResult("T1", "Manifest completeness and burden coverage", passed, metrics, criteria, "CF03 burden manifest verified."))


## Gate T2 — Audit-plan strength versus toy failure modes

**Plain-language view**

A notebook can have the right manifest and still be weak if all cells do the same kind of test.

This cell checks whether the instantiated CF03 audit actually includes multiple attack families:
- exact residuals
- sweeps / convergence tests
- negative controls or adversarial perturbations
- paper-level falsifier gates


In [ ]:
attack_modes = [str(c.get("attack_mode", "")).lower() for c in PAPER_SPEC["claims"]]
attack_text = " | ".join(attack_modes)

attack_family_hits = {
    "exact_residual": int("residual" in attack_text or "exact" in attack_text),
    "negative_control": int("negative control" in attack_text),
    "adversarial_or_perturbation": int("perturb" in attack_text or "adversarial" in attack_text),
    "sweep_or_convergence": int("sweep" in attack_text or "convergence" in attack_text or "time evolution" in attack_text or "fit residual" in attack_text),
    "paper_level_gate": int(len(PAPER_SPEC.get("paper_validation_gates", [])) > 0),
}
thresholdless_claims = [c["claim_id"] for c in PAPER_SPEC["claims"] if not c.get("pass_criteria")]
falsifier_like_claims = [c["claim_id"] for c in PAPER_SPEC["claims"] if c.get("claim_type") in {"falsifier", "scaling_prediction"}]

metrics = {
    "strong_attack_family_count": sum(attack_family_hits.values()),
    "negative_control_present": bool(attack_family_hits["negative_control"]),
    "thresholdless_claim_count": len(thresholdless_claims),
    "falsifier_like_claim_count": len(falsifier_like_claims),
}
criteria = {
    "strong_attack_family_count": ">= 4",
    "negative_control_present": True,
    "thresholdless_claim_count": 0,
    "paper_level_gate_present": True,
}

fig, ax = plt.subplots(figsize=(7.8, 3.3))
ax.bar(list(attack_family_hits.keys()), list(attack_family_hits.values()))
ax.set_ylim(0, 1.2)
ax.set_title("Audit-plan strength by attack family")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.25)
plt.show()

passed = (
    metrics["strong_attack_family_count"] >= 4
    and metrics["negative_control_present"]
    and metrics["thresholdless_claim_count"] == 0
    and attack_family_hits["paper_level_gate"] == 1
)
terminal_log(
    "T2",
    "Audit-plan strength versus toy failure modes",
    passed,
    metrics,
    criteria,
    notes=f"attack_family_hits={attack_family_hits}; falsifier_like_claims={falsifier_like_claims}",
)
LEDGER.append(GateResult("T2", "Audit-plan strength versus toy failure modes", passed, metrics, criteria, "CF03 audit uses multiple attack families."))


## Gate CF03-C1 — Sharp-interface bridge

**Plain-language view**

This cell attacks the bridge from diffuse phase-field energy to sharp-interface energy using the exact 1D heteroclinic profile.

**What would fail the claim**
- a resolved single interface does **not** carry energy approximately equal to \(c_0\)
- the residual stays macroscopic instead of shrinking to numerical noise

**Pass / fail criteria**
- maximum relative error across the epsilon sweep is at most \(5\times10^{-5}\)

**Intellectual honesty**
This is a **sanity attack on the advertised bridge**, not a re-proof of full \(\Gamma\)-convergence.


In [ ]:
eps_values = np.array([0.40, 0.25, 0.18, 0.12, 0.08, 0.05], dtype=float)
energies = []
rel_errors = []

for eps in eps_values:
    dx = eps / 64.0
    x = np.arange(-12.0, 12.0 + dx, dx)
    phi = heteroclinic_profile(x, eps)
    E, _, _ = phase_energy_1d(x, phi, eps)
    energies.append(E)
    rel_errors.append(abs(E - C0_EXACT) / C0_EXACT)

energies = np.array(energies)
rel_errors = np.array(rel_errors)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.semilogy(eps_values, rel_errors, marker="o")
ax.set_xlabel("epsilon")
ax.set_ylabel("relative error to c0")
ax.set_title("CF03-C1: single-interface energy vs c0")
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "c0_exact": float(C0_EXACT),
    "max_relative_error": float(np.max(rel_errors)),
    "min_relative_error": float(np.min(rel_errors)),
    "eps_values": eps_values.tolist(),
}
criteria = {
    "max_relative_error": "<= 5e-5",
}
passed = float(np.max(rel_errors)) <= 5e-5
terminal_log(
    "CF03-C1",
    "Sharp-interface bridge",
    passed,
    metrics,
    criteria,
    notes=f"energies={energies.tolist()}",
)
LEDGER.append(GateResult("CF03-C1", "Sharp-interface bridge", passed, metrics, criteria, "Resolved heteroclinic energies stay at c0 within tolerance."))


## Gate CF03-THM-MM — Modica–Mortola theorem sanity attack

**Plain-language view**

If the theorem is the right effective description, then several well-separated interfaces should cost energy proportional to the number of boundary points.

**What this cell attacks**
- nonlinearity in energy vs interface count
- wrong slope relative to \(c_0\)
- accidental fit quality without perimeter scaling

**Pass / fail criteria**
- fitted slope matches \(c_0\) to within \(10^{-3}\) relative error
- linear fit \(R^2 \ge 0.999\)

**Intellectual honesty**
This is not a substitute for the theorem. It is a direct numerical consistency check on the theorem's most exposed consequence in 1D.


In [ ]:
eps = 0.08
dx = eps / 64.0
L = 18.0
x = np.arange(-L, L + dx, dx)

interface_counts = np.arange(1, 7, dtype=int)
energies = []

for m in interface_counts:
    centers = np.linspace(-0.7 * L, 0.7 * L, m)
    phi = multi_interface_profile(x, centers.tolist(), eps)
    E, _, _ = phase_energy_1d(x, phi, eps)
    energies.append(E)

energies = np.array(energies)
fit = linear_fit_metrics(interface_counts.astype(float), energies)
slope_rel_error = abs(fit["slope"] - C0_EXACT) / C0_EXACT

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(interface_counts, energies, "o", label="numerical energies")
ax.plot(interface_counts, fit["yhat"], "-", label="linear fit")
ax.set_xlabel("boundary count / perimeter in 1D")
ax.set_ylabel("E_eps")
ax.set_title("CF03-THM-MM: energy linearity in boundary count")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "fitted_slope": float(fit["slope"]),
    "c0_exact": float(C0_EXACT),
    "slope_relative_error": float(slope_rel_error),
    "r2": float(fit["r2"]),
}
criteria = {
    "slope_relative_error": "<= 1e-3",
    "r2": ">= 0.999",
}
passed = slope_rel_error <= 1e-3 and fit["r2"] >= 0.999
terminal_log(
    "CF03-THM-MM",
    "Modica-Mortola theorem sanity attack",
    passed,
    metrics,
    criteria,
    notes=f"energies={energies.tolist()}",
)
LEDGER.append(GateResult("CF03-THM-MM", "Modica-Mortola theorem sanity attack", passed, metrics, criteria, "Energy remains linear in boundary count with slope c0."))


## Gate CF03-C2 — Boundary-energy concentration

**Plain-language view**

This cell checks whether the excess energy is actually localized near the interface instead of spread through the bulk.

**What would fail the claim**
- a significant fraction of excess energy remains outside an \(O(\varepsilon)\) neighborhood
- cumulative energy fraction rises too slowly with neighborhood width

**Pass / fail criteria**
- at least 98 percent of excess energy lies inside a \(2\varepsilon\) neighborhood

**Intellectual honesty**
This tests the localization profile on the canonical interface; it does not claim to exhaust every admissible recovery sequence.


In [ ]:
eps = 0.10
dx = eps / 80.0
x = np.arange(-12.0, 12.0 + dx, dx)
phi = heteroclinic_profile(x, eps)
E, density, _ = phase_energy_1d(x, phi, eps)

Ks = np.linspace(0.25, 3.0, 25)
fractions = []
for K in Ks:
    mask = np.abs(x) <= K * eps
    fractions.append(float(trapz(density[mask], x[mask])) / E)
fractions = np.array(fractions)

fraction_at_2eps = float(np.interp(2.0, Ks, fractions))

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(Ks, fractions, marker="o", ms=3)
ax.axvline(2.0, linestyle="--")
ax.axhline(0.98, linestyle="--")
ax.set_xlabel("neighborhood half-width / epsilon")
ax.set_ylabel("captured excess-energy fraction")
ax.set_title("CF03-C2: localization of excess energy near interface")
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "fraction_at_2eps": fraction_at_2eps,
    "fraction_at_1p5eps": float(np.interp(1.5, Ks, fractions)),
    "fraction_at_3eps": float(np.interp(3.0, Ks, fractions)),
}
criteria = {
    "fraction_at_2eps": ">= 0.98",
}
passed = fraction_at_2eps >= 0.98
terminal_log(
    "CF03-C2",
    "Boundary-energy concentration",
    passed,
    metrics,
    criteria,
    notes="Bulk energy stays negligible once the interface neighborhood reaches a few epsilon.",
)
LEDGER.append(GateResult("CF03-C2", "Boundary-energy concentration", passed, metrics, criteria, "Excess energy localizes near the interface."))


## Gate CF03-C3 — Logarithmic depth claim

**Plain-language view**

This cell attacks both sides of the claim:
- **upper side**: depth cannot exceed the available dyadic ladder
- **lower side / witness side**: a nested construction can still realize logarithmic growth

**Pass / fail criteria**
- no bound violations
- linear fit of \(N(L)\) versus \(\log_2(L/\ell_0)\) has \(R^2 \ge 0.98\)

**Intellectual honesty**
This is a constructive scaling attack, not a proof that all dynamics realize the witness.


In [ ]:
ratios = np.array([3, 5, 7, 9, 13, 17, 33, 65, 129, 257], dtype=float)
N = np.floor(np.log2(ratios)).astype(int) + 1
upper_bound = np.ceil(np.log2(ratios)).astype(int) + 1
violations = int(np.sum(N > upper_bound))

fit = fit_log_model(ratios, N.astype(float))

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(np.log2(ratios), N, "o", label="witness depth")
ax.plot(np.log2(ratios), fit["yhat"], "-", label="best linear fit")
ax.set_xlabel("log2(L / ell0)")
ax.set_ylabel("N(L)")
ax.set_title("CF03-C3: logarithmic witness for active depth")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "bound_violations": violations,
    "log_fit_r2": float(fit["r2"]),
    "fitted_slope": float(fit["coef"][0]),
    "fitted_intercept": float(fit["coef"][1]),
}
criteria = {
    "bound_violations": 0,
    "log_fit_r2": ">= 0.98",
}
passed = violations == 0 and fit["r2"] >= 0.98
terminal_log(
    "CF03-C3",
    "Logarithmic depth claim",
    passed,
    metrics,
    criteria,
    notes=f"upper_bound={upper_bound.tolist()} ; N={N.tolist()}",
)
LEDGER.append(GateResult("CF03-C3", "Logarithmic depth claim", passed, metrics, criteria, "Witness depth remains logarithmic and never outruns the dyadic ladder."))


## Gate CF03-PROP-LOG — Proposition: dyadic upper bound

**Plain-language view**

The proposition says something very hard to excuse away: if active depth is counted on a dyadic ladder, it can never exceed the number of available dyadic scales.

**What this cell attacks**
- hidden off-by-one errors
- bad depth definitions that accidentally overcount
- hand-wavy appeals to “logarithmic” without exact ladder accounting

**Pass / fail criteria**
- zero violations across exhaustive enumeration of dyadic activity masks up to ladder length 10


In [ ]:
max_ladder_length = 10
ladder_lengths = np.arange(1, max_ladder_length + 1, dtype=int)
max_depths = []
violations = 0

for n in ladder_lengths:
    local_max = 0
    for mask_int in range(2**n):
        bits = np.array([(mask_int >> j) & 1 for j in range(n)], dtype=int)
        depth = int(np.sum(bits))
        local_max = max(local_max, depth)
        if depth > n:
            violations += 1
    max_depths.append(local_max)

max_depths = np.array(max_depths)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(ladder_lengths, max_depths, "o-", label="max observed depth")
ax.plot(ladder_lengths, ladder_lengths, "--", label="ladder-length ceiling")
ax.set_xlabel("dyadic ladder length")
ax.set_ylabel("max possible active depth")
ax.set_title("CF03-PROP-LOG: exhaustive dyadic bound check")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "violations": int(violations),
    "tested_masks_total": int(np.sum(2**ladder_lengths)),
    "max_depths": max_depths.tolist(),
}
criteria = {
    "violations": 0,
}
passed = violations == 0
terminal_log(
    "CF03-PROP-LOG",
    "Proposition: dyadic upper bound",
    passed,
    metrics,
    criteria,
    notes="Exhaustive enumeration checks the exact combinatorial ceiling, not just sampled cases.",
)
LEDGER.append(GateResult("CF03-PROP-LOG", "Proposition: dyadic upper bound", passed, metrics, criteria, "No dyadic activity mask exceeds its ladder length."))


## Gate CF03-C4 — No combinatorial blow-up under perimeter-reducing relaxation

**Plain-language view**

This cell attacks the empirical claim on a deliberately hierarchical initial condition. If the claim is directionally right, Allen–Cahn relaxation should not invent new same-scale multiplicity out of nowhere.

**What would fail the claim**
- per-scale active boundary counts increase above their initial values
- diffuse energy increases during relaxation

**Pass / fail criteria**
- maximum positive increase in any measured per-scale count is zero
- diffuse energy is nonincreasing over the sampled trajectory

**Intellectual honesty**
This is an empirical attack on a 1D hierarchical initial condition. It is support, not a universal theorem.


In [ ]:
N_grid = 1024
L = 1.0
x = np.linspace(0.0, L, N_grid, endpoint=False)
dx = x[1] - x[0]
eps = 0.01
ell0 = 1.0 / 64.0
centers = [0.10, 0.27, 0.48, 0.74]

phi = multi_interface_profile(x, centers, eps)
dt = 0.1 * dx * dx / eps**2
sample_times = [0, 200, 800, 2000, 4000]

sample_counts = []
sample_energies = []

for t in range(max(sample_times) + 1):
    if t in sample_times:
        _, counts = active_boundary_census(phi, L=L, ell0=ell0, dx=dx, sigma_frac=0.20)
        sample_counts.append(counts.copy())
        density = 0.5 * eps * ((np.roll(phi, -1) - phi) / dx) ** 2 + W(phi) / eps
        sample_energies.append(float(np.sum(density) * dx))
    phi = allen_cahn_step(phi, dx=dx, dt=dt, eps=eps)

sample_counts = np.array(sample_counts)
sample_energies = np.array(sample_energies)
count_increase = sample_counts - sample_counts[0]
max_positive_count_increase = int(np.max(count_increase))
energy_nonincreasing = bool(np.all(np.diff(sample_energies) <= 1e-10))

fig, ax = plt.subplots(figsize=(7.6, 3.5))
for j in range(sample_counts.shape[1]):
    ax.plot(sample_times, sample_counts[:, j], marker="o", label=f"k={j}")
ax.set_xlabel("Allen-Cahn steps")
ax.set_ylabel("active boundary count at scale k")
ax.set_title("CF03-C4: per-scale counts under perimeter-reducing relaxation")
ax.grid(alpha=0.25)
ax.legend(ncol=4, fontsize=8)
plt.show()

metrics = {
    "initial_counts": sample_counts[0].tolist(),
    "final_counts": sample_counts[-1].tolist(),
    "max_positive_count_increase": max_positive_count_increase,
    "energy_nonincreasing": energy_nonincreasing,
    "sample_energies": sample_energies.tolist(),
}
criteria = {
    "max_positive_count_increase": 0,
    "energy_nonincreasing": True,
}
passed = max_positive_count_increase == 0 and energy_nonincreasing
terminal_log(
    "CF03-C4",
    "No combinatorial blow-up under perimeter-reducing relaxation",
    passed,
    metrics,
    criteria,
    notes="The trajectory is intentionally scoped: hierarchical initial condition, 1D Allen-Cahn, finite observation window.",
)
LEDGER.append(GateResult("CF03-C4", "No combinatorial blow-up under perimeter-reducing relaxation", passed, metrics, criteria, "Per-scale multiplicities did not amplify during the sampled relaxation."))


## Gate CF03-G1 — Perimeter proxy scaling falsifier

**Plain-language view**

A good falsifier should not just pass the intended proxy. It should also reject an obviously wrong proxy.

**What this cell attacks**
- false positives from arbitrary regressions
- accidental linearity with a non-perimeter observable

**Pass / fail criteria**
- mean relative error for the perimeter proxy is at most 5 percent
- a bad proxy is materially worse and therefore rejected


In [ ]:
eps = 0.08
dx = eps / 64.0
L = 18.0
x = np.arange(-L, L + dx, dx)

ms = np.arange(1, 7, dtype=int)
energies = []
perim_proxy = []
area_proxy = []

for m in ms:
    centers = np.linspace(-0.7 * L, 0.7 * L, m)
    phi = multi_interface_profile(x, centers.tolist(), eps)
    E, _, _ = phase_energy_1d(x, phi, eps)
    energies.append(E)
    perim_proxy.append(perimeter_proxy_open(phi))
    area_proxy.append(float(np.mean(phi > 0.0)))

energies = np.array(energies)
perim_proxy = np.array(perim_proxy, dtype=float)
area_proxy = np.array(area_proxy, dtype=float)

fit_good = linear_fit_metrics(perim_proxy, energies)
fit_bad = linear_fit_metrics(area_proxy, energies)

mre_good = float(np.mean(np.abs(energies - fit_good["yhat"]) / energies))
mre_bad = float(np.mean(np.abs(energies - fit_bad["yhat"]) / energies))
negative_control_rejected = bool(mre_bad > mre_good * 10.0)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(perim_proxy, energies, "o", label="energy vs perimeter proxy")
ax.plot(perim_proxy, fit_good["yhat"], "-", label="fit using perimeter proxy")
ax.plot(area_proxy, fit_bad["yhat"], "s--", label="fit using bad area proxy")
ax.set_xlabel("proxy value")
ax.set_ylabel("E_eps")
ax.set_title("CF03-G1: perimeter proxy vs bad proxy")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "mean_relative_error_good_proxy": mre_good,
    "mean_relative_error_bad_proxy": mre_bad,
    "negative_control_rejected": negative_control_rejected,
}
criteria = {
    "mean_relative_error_good_proxy": "<= 0.05",
    "negative_control_rejected": True,
}
passed = mre_good <= 0.05 and negative_control_rejected
terminal_log(
    "CF03-G1",
    "Perimeter proxy scaling falsifier",
    passed,
    metrics,
    criteria,
    notes="The bad proxy is intentionally unrelated to boundary count, so the gate must separate it cleanly.",
)
LEDGER.append(GateResult("CF03-G1", "Perimeter proxy scaling falsifier", passed, metrics, criteria, "Perimeter proxy passes; bad area proxy is rejected."))


## Gate CF03-G2 — No per-scale exponential growth falsifier

**Plain-language view**

This gate only earns trust if it can separate two deliberately different sequences:
- a bounded hierarchical sequence
- an adversarial exponential sequence

**Pass / fail criteria**
- bounded hierarchical multiplicities pass the gate
- exponential multiplicities fail the gate


In [ ]:
k = np.arange(0, 8, dtype=int)
hierarchical_counts = np.ones_like(k, dtype=float)
exponential_counts = 2.0 ** k
m_max = 2.0

hierarchical_passes = bool(np.all(hierarchical_counts <= m_max))
exponential_fails = bool(np.any(exponential_counts > m_max))

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(k, hierarchical_counts, "o-", label="bounded hierarchy")
ax.plot(k, exponential_counts, "s--", label="exponential blow-up")
ax.axhline(m_max, linestyle="--", label="m_max")
ax.set_xlabel("dyadic scale index k")
ax.set_ylabel("active multiplicity m_k")
ax.set_title("CF03-G2: gate separation on bounded vs exponential multiplicity")
ax.set_yscale("log")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "m_max": float(m_max),
    "hierarchical_passes": hierarchical_passes,
    "exponential_fails": exponential_fails,
}
criteria = {
    "hierarchical_passes": True,
    "exponential_fails": True,
}
passed = hierarchical_passes and exponential_fails
terminal_log(
    "CF03-G2",
    "No per-scale exponential growth falsifier",
    passed,
    metrics,
    criteria,
    notes="This is a gate-separation attack on the falsifier itself: it must not confuse bounded hierarchy with exponential multiplicity.",
)
LEDGER.append(GateResult("CF03-G2", "No per-scale exponential growth falsifier", passed, metrics, criteria, "Gate separates bounded hierarchy from exponential blow-up."))


## Gate CF03-G3 — Depth scaling and model selection falsifier

**Plain-language view**

A real falsifier should:
- choose the logarithmic law on logarithmic data
- reject the logarithmic law on power-law data

**Pass / fail criteria**
- the log-depth dataset prefers the log model by both AIC and BIC
- the power-law negative control prefers the power model by both AIC and BIC


In [ ]:
## Gate CF03-G3 — Depth scaling and model selection falsifier

# Plain-language view
# A real falsifier should:
# - choose the logarithmic law on logarithmic data
# - reject the logarithmic law on power-law data

# Pass / fail criteria
# - the log-depth dataset prefers the log model by both AIC and BIC
# - the power-law negative control prefers the power model by both AIC and BIC

ratio = 2.0 ** np.arange(2, 10)  # dyadic sweep: 4, 8, 16, ..., 512

# Positive control for THIS gate: true logarithmic law
N_log = np.log2(ratio) + 1.0

# Negative control: true power law
N_power = 0.9 * ratio**0.35

log_fit_on_log = fit_log_model(ratio, N_log)
pow_fit_on_log = fit_power_model(ratio, N_log)

log_fit_on_power = fit_log_model(ratio, N_power)
pow_fit_on_power = fit_power_model(ratio, N_power)

delta_aic_log_dataset = pow_fit_on_log["aic"] - log_fit_on_log["aic"]
delta_bic_log_dataset = pow_fit_on_log["bic"] - log_fit_on_log["bic"]
delta_aic_power_dataset = log_fit_on_power["aic"] - pow_fit_on_power["aic"]
delta_bic_power_dataset = log_fit_on_power["bic"] - pow_fit_on_power["bic"]

log_dataset_prefers_log = bool(
    delta_aic_log_dataset > 0.0
    and delta_bic_log_dataset > 0.0
    and log_fit_on_log["r2"] >= 0.98
)

power_dataset_rejects_log = bool(
    delta_aic_power_dataset > 0.0
    and delta_bic_power_dataset > 0.0
)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.bar(
    ["log-data ΔAIC", "log-data ΔBIC", "power-data ΔAIC", "power-data ΔBIC"],
    [
        delta_aic_log_dataset,
        delta_bic_log_dataset,
        delta_aic_power_dataset,
        delta_bic_power_dataset,
    ],
)
ax.axhline(0.0, linestyle="--")
ax.set_ylabel("positive = correct model preferred")
ax.set_title("CF03-G3: model-selection separation")
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "log_fit_r2_on_log_dataset": float(log_fit_on_log["r2"]),
    "delta_aic_log_dataset": float(delta_aic_log_dataset),
    "delta_bic_log_dataset": float(delta_bic_log_dataset),
    "delta_aic_power_dataset": float(delta_aic_power_dataset),
    "delta_bic_power_dataset": float(delta_bic_power_dataset),
    "log_dataset_prefers_log": log_dataset_prefers_log,
    "power_dataset_rejects_log": power_dataset_rejects_log,
}
criteria = {
    "log_dataset_prefers_log": True,
    "power_dataset_rejects_log": True,
}
passed = log_dataset_prefers_log and power_dataset_rejects_log

terminal_log(
    "CF03-G3",
    "Depth scaling and model selection falsifier",
    passed,
    metrics,
    criteria,
    notes=(
        "Positive bars mean the intended model beat the alternative. "
        "This gate uses a true logarithmic positive control; the dyadic staircase "
        "claim is audited elsewhere."
    ),
)

LEDGER.append(
    GateResult(
        "CF03-G3",
        "Depth scaling and model selection falsifier",
        passed,
        metrics,
        criteria,
        "Model selection cleanly separates logarithmic and power-law depth laws.",
    )
)

## Gate CF03-G4 — Dimensionless collapse falsifier

**Plain-language view**

If CF03 is really making a scale claim, then data with the same \(L/\ell_0\) should collapse even when \(L\) and \(\ell_0\) change separately. A deliberately contaminated dataset should fail that collapse.

**Pass / fail criteria**
- true collapse error at most 0.02
- false collapse error at least 0.08


In [ ]:
ratio = np.array([4, 6, 8, 12, 16, 24, 32, 48, 64], dtype=float)
ell0_values = np.array([1.0, 2.0, 4.0], dtype=float)

true_curves = []
false_curves = []

for ell0 in ell0_values:
    L = ratio * ell0
    true_obs = np.log1p(L / ell0) / np.sqrt(L / ell0)
    false_obs = true_obs + 0.15 * np.log(ell0)
    true_curves.append(true_obs)
    false_curves.append(false_obs)

true_curves = np.array(true_curves)
false_curves = np.array(false_curves)

true_collapse_error = float(np.mean(np.std(true_curves, axis=0)) / np.mean(np.abs(true_curves)))
false_collapse_error = float(np.mean(np.std(false_curves, axis=0)) / np.mean(np.abs(false_curves)))

fig, ax = plt.subplots(figsize=(7.2, 3.6))
for i, ell0 in enumerate(ell0_values):
    ax.plot(ratio, true_curves[i], marker="o", label=f"true ell0={ell0:g}")
for i, ell0 in enumerate(ell0_values):
    ax.plot(ratio, false_curves[i], linestyle="--", alpha=0.7, label=f"false ell0={ell0:g}")
ax.set_xlabel("L / ell0")
ax.set_ylabel("observable")
ax.set_title("CF03-G4: true collapse vs contaminated non-collapse")
ax.grid(alpha=0.25)
ax.legend(ncol=2, fontsize=8)
plt.show()

metrics = {
    "true_collapse_error": true_collapse_error,
    "false_collapse_error": false_collapse_error,
}
criteria = {
    "true_collapse_error": "<= 0.02",
    "false_collapse_error": ">= 0.08",
}
passed = true_collapse_error <= 0.02 and false_collapse_error >= 0.08
terminal_log(
    "CF03-G4",
    "Dimensionless collapse falsifier",
    passed,
    metrics,
    criteria,
    notes="Solid curves are genuinely dimensionless; dashed curves contain injected separate ell0 dependence.",
)
LEDGER.append(GateResult("CF03-G4", "Dimensionless collapse falsifier", passed, metrics, criteria, "Dimensionless collapse survives only on the uncontaminated dataset."))


## Gate CF03-W1 — Worked witness: exact surface tension constant

**Plain-language view**

This is the simplest exact burden in the worked example: does the quartic double-well really give \(c_0 = 2\sqrt{2}/3\)?

**Pass / fail criteria**
- numerical quadrature agrees with the closed form to within \(10^{-10}\)

**Intellectual honesty**
This is a worked witness, not a theorem beyond the stated potential.


In [ ]:
s = np.linspace(-1.0, 1.0, 200001)
integrand = np.sqrt(0.5 * (1.0 - s**2) ** 2)
c0_numeric = float(trapz(integrand, s))
abs_error = abs(c0_numeric - C0_EXACT)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(s, integrand)
ax.set_xlabel("s")
ax.set_ylabel("sqrt(1/2 * (1-s^2)^2)")
ax.set_title("CF03-W1: integrand defining c0")
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "c0_exact": float(C0_EXACT),
    "c0_numeric": c0_numeric,
    "absolute_error": float(abs_error),
}
criteria = {
    "absolute_error": "<= 1e-10",
}
passed = abs_error <= 1e-10
terminal_log(
    "CF03-W1",
    "Worked witness: exact surface tension constant",
    passed,
    metrics,
    criteria,
    notes="This is the exact worked-example constant used later in the 1D witness section.",
)
LEDGER.append(GateResult("CF03-W1", "Worked witness: exact surface tension constant", passed, metrics, criteria, "Quartic-potential surface tension constant matches closed form."))


## Gate CF03-W2 — Worked witness: nested construction and logarithmic energy

**Plain-language view**

This is the paper's constructive witness: one interface per dyadic scale up to the cutoff should give logarithmic depth and energy proportional to that depth.

**Pass / fail criteria**
- fitted slope of \(E_0\) versus \(\log_2(L/\ell_0)\) matches \(c_0\) essentially exactly
- \(R^2 \ge 0.999999\)

**Intellectual honesty**
This is a realizability witness, exactly as the paper states. It is not a claim about typical dynamics.


In [ ]:
ratio = 2.0 ** np.arange(1, 10, dtype=float)
N_depth = np.floor(np.log2(ratio)).astype(float) + 1.0
E0 = C0_EXACT * N_depth

fit = fit_log_model(ratio, E0)
slope_rel_error = abs(float(fit["coef"][0]) - C0_EXACT) / C0_EXACT

fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.plot(np.log2(ratio), E0, "o", label="constructive witness")
ax.plot(np.log2(ratio), fit["yhat"], "-", label="best linear fit")
ax.set_xlabel("log2(L / ell0)")
ax.set_ylabel("E0")
ax.set_title("CF03-W2: logarithmic energy of dyadic witness")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

metrics = {
    "fitted_slope": float(fit["coef"][0]),
    "c0_exact": float(C0_EXACT),
    "slope_relative_error": float(slope_rel_error),
    "r2": float(fit["r2"]),
}
criteria = {
    "slope_relative_error": "<= 1e-12",
    "r2": ">= 0.999999",
}
passed = slope_rel_error <= 1e-12 and fit["r2"] >= 0.999999
terminal_log(
    "CF03-W2",
    "Worked witness: nested construction and logarithmic energy",
    passed,
    metrics,
    criteria,
    notes="The witness is exact on powers of two because the depth law is constructed on the dyadic ladder.",
)
LEDGER.append(GateResult("CF03-W2", "Worked witness: nested construction and logarithmic energy", passed, metrics, criteria, "Constructive dyadic witness reproduces logarithmic energy with slope c0."))


## Final ledger requirement

The notebook must end with an explicit ledger. Missing or failed burden units should remain visible here rather than being smoothed away.


## Gate T6 — Notebook-wide final results ledger

**Plain-language view**

This final cell checks the integrity of the ledger itself and then prints the full table.


In [ ]:
gate_ids = [g.gate_id for g in LEDGER]
pass_count = sum(int(g.passed) for g in LEDGER)
fail_count = len(LEDGER) - pass_count

fig, ax = plt.subplots(figsize=(9.0, 3.6))
ax.bar(gate_ids, [1 if g.passed else 0 for g in LEDGER])
ax.set_ylim(0, 1.2)
ax.set_title("Final results ledger")
ax.set_ylabel("PASS = 1, FAIL = 0")
ax.tick_params(axis="x", rotation=35)
ax.grid(alpha=0.25)
plt.show()

metrics = {
    "ledger_gate_count": len(LEDGER),
    "unique_gate_count": len(set(gate_ids)),
    "pass_count": pass_count,
    "fail_count": fail_count,
}
criteria = {
    "ledger_gate_count": ">= 1",
    "unique_gate_count_equals_ledger_gate_count": True,
    "pass_count_plus_fail_count_equals_ledger_gate_count": True,
}
passed = (
    len(LEDGER) >= 1
    and len(set(gate_ids)) == len(LEDGER)
    and (pass_count + fail_count == len(LEDGER))
)
terminal_log(
    "T6",
    "Notebook-wide final results ledger",
    passed,
    metrics,
    criteria,
    notes="This ledger is supposed to stay blunt: unresolved failures should remain visible.",
)
LEDGER.append(GateResult("T6", "Notebook-wide final results ledger", passed, metrics, criteria, "Final CF03 ledger emitted."))

print("\nFINAL LEDGER TABLE")
print("-" * 100)
for g in LEDGER:
    print(f"{g.gate_id:>12} | {'PASS' if g.passed else 'FAIL':<4} | {g.gate_name}")
print("-" * 100)
print(f"TOTAL: {len(LEDGER)} gates | PASS={sum(int(g.passed) for g in LEDGER)} | FAIL={len(LEDGER)-sum(int(g.passed) for g in LEDGER)}")


## Publication checklist for this instantiated notebook

Before treating this notebook as complete, verify:
- every burden unit you care about is present in the ledger
- no helper/setup cell violates the figure + gate-log contract
- empirical support cells are not silently re-labeled as proofs
- failed gates, if any, stay visible instead of being buried in prose
